# U.S. Patent Phrase to Phrase Matching
## A Comprehensive Learning & Solution Notebook

---

### About This Notebook
This notebook teaches Data Science concepts while solving a real Kaggle competition.

### What You'll Learn
1. Data Processing with Pandas
2. Text Embeddings with Sentence-BERT
3. Deep Learning with DeBERTa
4. Model Evaluation with Pearson Correlation

### Competition
- **Goal**: Predict similarity (0-1) between patent phrases
- **Metric**: Pearson Correlation Coefficient
- **Expected Score**: ~0.80 with DeBERTa fine-tuning

### License: MIT - Feel free to share and learn!

---


In [ ]:
# ============================================================
# SECTION 1: SETUP AND IMPORTS
# ============================================================

import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr

import torch

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

print("=" * 60)
print("LIBRARIES LOADED")
print("=" * 60)
print(f"PyTorch: {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Setup complete!")

In [ ]:
# ============================================================
# SECTION 2: HUGGINGFACE AUTHENTICATION (OPTIONAL)
# ============================================================

import os

HF_TOKEN = os.environ.get("HF_TOKEN", None)

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged into HuggingFace!")
else:
    print("No HF_TOKEN found - using unauthenticated (slower)")
    print("Set: export HF_TOKEN='hf_xxx' for faster downloads")

---

## Section 3: Load Data

### Data Format
- **train.csv**: Contains anchor, target, context, score
- **test.csv**: Contains anchor, target, context (no score)
- **score**: 0 = dissimilar, 1 = identical

---


In [ ]:
import glob
# ============================================================
# SECTION 3: LOAD DATA
# ============================================================

OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Auto-detect data path for Kaggle vs local
possible_paths = [
    "/kaggle/input/competitions/us-patent-phrase-to-phrase-matching",  # Kaggle competition data (new format)
    "/kaggle/input/us-patent-phrase-to-phrase-matching",  # Kaggle competition data (old format)
    "/kaggle/input",
    "./data",  # Local development
    "data",
]

# Robust path detection - search multiple locations
search_patterns = [
    "/kaggle/input/datasets/*/us-patent-phrase-to-phrase-matching",
    "/kaggle/input/datasets/*/*patent*phrase*",
    "/kaggle/input/competitions/us-patent-phrase-to-phrase-matching",
    "/kaggle/input/us-patent-phrase-to-phrase-matching",
    "/kaggle/input",
    "./data",
    "data",
]

DATA_DIR = None
for pattern in search_patterns:
    matches = glob.glob(pattern)
    for match in matches:
        if os.path.exists(f"{match}/train.csv"):
            DATA_DIR = match
            break
    if DATA_DIR:
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        f"Could not find train.csv. Please add competition data to notebook."
    )

print(f"Data directory: {DATA_DIR}")
print("Loading data...")
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Columns: {train_df.columns.tolist()}")

print("\nSample data:")
display(train_df.head())


---

## Section 4: SBERT Baseline (No GPU Needed)

### What is Sentence-BERT?
- Pre-trained model for sentence embeddings
- Converts text to 768-dimensional vectors
- We compute cosine similarity between vectors

### Expected Performance
- Pearson: ~0.62
- Fast: runs without GPU

---


In [ ]:
# ============================================================
# SECTION 4: SBERT BASELINE
# ============================================================

from sentence_transformers import SentenceTransformer

print("=" * 60)
print("SBERT BASELINE")
print("=" * 60)

model_name = 'all-mpnet-base-v2'
print(f"Loading model: {model_name}")
sbert_model = SentenceTransformer(model_name)
print(f"Model loaded! Dimension: {sbert_model.get_sentence_embedding_dimension()}")

# Generate embeddings for validation
train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=42)
print(f"\nValidation samples: {len(val_split)}")

print("Encoding validation data...")
val_anchor_emb = sbert_model.encode(val_split['anchor'].tolist(), show_progress_bar=True)
val_target_emb = sbert_model.encode(val_split['target'].tolist(), show_progress_bar=True)

# Compute similarity scores
val_scores = []
for i in range(len(val_split)):
    sim = cosine_similarity(val_anchor_emb[i:i+1], val_target_emb[i:i+1])[0, 0]
    val_scores.append(sim)

val_scores = np.array(val_scores)

# Evaluate
actual_scores = val_split['score'].values
pearson_corr, _ = pearsonr(val_scores, actual_scores)

print(f"\nValidation Pearson Correlation: {pearson_corr:.4f}")
print("SBERT baseline complete!")

---

## Section 5: DeBERTa Fine-tuning (GPU Required)

### What is DeBERTa?
- Microsoft's state-of-the-art transformer model
- Disentangled attention mechanism
- Fine-tuned for our specific task

### Expected Performance
- Pearson: ~0.78-0.82
- Requires GPU
- Takes ~60-90 minutes

### Configuration
- Model: microsoft/deberta-v3-small
- Learning rate: 2e-5
- Epochs: 3
- Batch size: 16

---


In [ ]:
# ============================================================
# SECTION 5: DEBERTA FINE-TUNING (GPU REQUIRED)
# ============================================================

print("=" * 60)
print("DEBERTA FINE-TUNING")
print("=" * 60)

if device != 'cuda':
    print("WARNING: GPU not available!")
    print("DeBERTa training requires GPU.")
    print("Enable GPU in notebook settings.")
else:
    print(f"GPU available: {torch.cuda.get_device_name(0)}")

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset

cfg = {
    'model_name': 'microsoft/deberta-v3-small',
    'max_length': 256,
    'batch_size': 16,
    'learning_rate': 2e-5,
    'num_epochs': 3,
}

print(f"Configuration: {cfg}")

# Prepare data
def prepare_inputs(df):
    return [f"{row['anchor']} [SEP] {row['target']} [SEP] {row['context']}" 
            for _, row in df.iterrows()]

tokenizer = AutoTokenizer.from_pretrained(cfg['model_name'])

# Split data into train/validation
from sklearn.model_selection import train_test_split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    prepare_inputs(train_df), 
    train_df['score'].tolist(),
    test_size=0.2,
    random_state=42
)

print(f"Train samples: {len(train_texts)}, Validation samples: {len(val_texts)}")

# Create datasets
train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})
val_dataset = Dataset.from_dict({'text': val_texts, 'label': val_labels})

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', 
                     truncation=True, max_length=cfg['max_length'])

train_dataset = train_dataset.map(tokenize_function, batched=True)
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

val_dataset = val_dataset.map(tokenize_function, batched=True)
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# Load model and ensure FP32
model = AutoModelForSequenceClassification.from_pretrained(
    cfg['model_name'], num_labels=1, problem_type="regression"
)
model = model.float()  # Ensure model weights are FP32

# Training arguments - optimized for Kaggle GPU stability
training_args = TrainingArguments(
    output_dir='./deberta_results',
    num_train_epochs=cfg['num_epochs'],
    per_device_train_batch_size=cfg['batch_size'],
    per_device_eval_batch_size=cfg['batch_size'] * 2,
    learning_rate=cfg['learning_rate'],
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none',
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

# Metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.flatten()
    corr, _ = pearsonr(predictions, labels)
    return {'pearson': corr}

# Trainer - now with eval_dataset
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()
print("Training complete!")


---

## Section 6: Generate Test Predictions

### This section creates the submission file
- Uses DeBERTa predictions if available
- Falls back to SBERT baseline
- Saves to output/submission.csv

---


In [ ]:
# ============================================================
# SECTION 6: GENERATE TEST PREDICTIONS
# ============================================================

print("=" * 60)
print("GENERATING TEST PREDICTIONS")
print("=" * 60)

# Check if we trained DeBERTa or use SBERT
if 'trainer' in dir():
    print("Using DeBERTa predictions...")
    
    # Prepare test data
    test_texts = prepare_inputs(test_df)
    test_dataset = Dataset.from_dict({'text': test_texts})
    test_dataset = test_dataset.map(tokenize_function, batched=True)
    test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])
    
    # Predict
    predictions = trainer.predict(test_dataset)
    test_scores = predictions.predictions.flatten()
    
    # Clip to valid range
    test_scores = np.clip(test_scores, 0, 1)
    
else:
    print("Using SBERT baseline predictions...")
    
    # Generate SBERT embeddings for test data
    test_anchor_emb = sbert_model.encode(test_df['anchor'].tolist(), show_progress_bar=True)
    test_target_emb = sbert_model.encode(test_df['target'].tolist(), show_progress_bar=True)
    
    # Compute similarity
    test_scores = []
    for i in range(len(test_df)):
        sim = cosine_similarity(test_anchor_emb[i:i+1], test_target_emb[i:i+1])[0, 0]
        test_scores.append(sim)
    test_scores = np.array(test_scores)

print(f"Generated {len(test_scores)} predictions")
print(f"Score range: [{test_scores.min():.4f}, {test_scores.max():.4f}]")

In [ ]:
# ============================================================
# SECTION 7: CREATE AND SAVE SUBMISSION
# ============================================================

print("=" * 60)
print("CREATING SUBMISSION")
print("=" * 60)

submission = pd.DataFrame({
    'id': test_df['id'],
    'score': test_scores
})

print("\nSubmission preview:")
display(submission.head(10))

# Verify format matches sample
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
print(f"\nFormat check:")
print(f"  Columns match: {submission.columns.tolist() == sample_sub.columns.tolist()}")
print(f"  Row count match: {len(submission) == len(sample_sub)}")

# Save submission
output_path = f"{OUTPUT_DIR}/submission.csv"
submission.to_csv(output_path, index=False)

print(f"\nSubmission saved to: {output_path}")

# Verify saved file
saved = pd.read_csv(output_path)
print(f"\nVerification:")
print(f"  Rows: {len(saved)}")
print(f"  First ID: {saved['id'].iloc[0]}")
print(f"  First score: {saved['score'].iloc[0]:.4f}")

print("\n" + "=" * 60)
print("SUBMISSION COMPLETE!")
print("=" * 60)
print(f"\nFile: {output_path}")
print("Ready to upload to Kaggle!")

---

## Summary

### What We Accomplished
1. Loaded and explored patent similarity data
2. Built SBERT baseline (~0.62 Pearson)
3. Fine-tuned DeBERTa (~0.80 Pearson with GPU)
4. Generated valid submission.csv

### Expected Scores
| Model | Pearson | GPU Needed |
|-------|---------|------------|
| SBERT Baseline | ~0.62 | No |
| DeBERTa Fine-tuned | ~0.80 | Yes |

### Files Generated
- output/submission.csv

### Learn More
- HuggingFace: huggingface.co/learn
- SBERT: sbert.net
- Kaggle: kaggle.com/competitions/us-patent-phrase-to-phrase-matching

---

**MIT License - Feel free to share and learn!**
